# 02 · SIFT Keypoints & Match Verification

**Goal:** detect SIFT keypoints across the extracted frames, match adjacent
frame pairs with Lowe's ratio test, and verify with cv2's RANSAC. The output
of this notebook is what feeds the from-scratch two-view reconstruction in
notebook 03.

Library-wrapped (cv2.SIFT, cv2.findEssentialMat) — geometry from scratch
lives in `03_two_view_scratch.ipynb`.

In [ ]:
# Standard preamble — every notebook seeds np.random.seed(131).
import sys
from pathlib import Path

# Make `src/` importable from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
np.random.seed(131)

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src import calibration, features, viz
import cv2

TREE_ID = "tree_oak_01"
FRAMES_DIR = PROJECT_ROOT / "data" / "frames" / TREE_ID
K_PATH = PROJECT_ROOT / "data" / "calibration" / "intrinsics.npz"


if not FRAMES_DIR.exists():
    raise FileNotFoundError(
        f"Frames directory not found at {FRAMES_DIR}.\n"
        "Run notebook 01 first to extract frames for this tree."
    )


if not K_PATH.exists():
    raise FileNotFoundError(
        f"Camera intrinsics not found at {K_PATH}.\n"
        "Run notebook 01 first to calibrate the camera."
    )


intrinsics = calibration.CameraIntrinsics.load(K_PATH)
frame_paths = sorted(FRAMES_DIR.glob("frame_*.png"))
print(f"{len(frame_paths)} frames; K loaded with RMS={intrinsics.rms:.3f} px.")


## 1. Pick a frame pair with strong parallax

For two-view reconstruction we want frames separated by ~1/4 of the orbit:
big enough baseline to triangulate, small enough overlap to share most
keypoints. With ~60 frames and a single orbit, that's roughly indices 0
and 15.

In [ ]:
IDX_A, IDX_B = 0, len(frame_paths) // 4
print(f"Frame A: {frame_paths[IDX_A].name}")
print(f"Frame B: {frame_paths[IDX_B].name}")

## 2. Detect SIFT + ratio-test match + RANSAC-verify

In [ ]:
feats_a, feats_b, pair = features.match_pair(
    frame_paths[IDX_A], frame_paths[IDX_B], K=intrinsics.K,
)
n_total = len(pair.pts1)
n_inlier = int(pair.inlier_mask.sum())
print(f"SIFT keypoints: A={len(feats_a.keypoints)}, B={len(feats_b.keypoints)}")
print(f"Ratio-test matches: {n_total}")
print(f"RANSAC inliers:     {n_inlier}  ({100*n_inlier/n_total:.1f}%)")

## 3. Visualise inliers vs outliers

Green lines are RANSAC inliers, red are outliers. Healthy pairs look like
a clean fan of green; lots of red suggests pure rotation or insufficient
baseline.

In [ ]:
img_a = cv2.imread(str(frame_paths[IDX_A]))
img_b = cv2.imread(str(frame_paths[IDX_B]))
fig = viz.plot_matches(
    img_a, img_b, pair.pts1, pair.pts2, pair.inlier_mask,
    title=f"{TREE_ID}: frames {IDX_A} vs {IDX_B} — {n_inlier}/{n_total} inliers",
)
viz.save_fig(fig, f"02_{TREE_ID}_matches.png")
plt.show()

## 4. Persist for the next stage

Notebook 03 will use these correspondences as the input to the from-scratch
two-view pipeline.

In [ ]:
cache_path = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_pair_{IDX_A}_{IDX_B}.npz"
cache_path.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    cache_path,
    pts1=pair.pts1[pair.inlier_mask],
    pts2=pair.pts2[pair.inlier_mask],
    K=intrinsics.K,
    E_ref=pair.E if pair.E is not None else np.zeros((3, 3)),
)
print(f"Cached inlier correspondences → {cache_path}")